# 🔧 Rust & Corrosion Detection with Severity Grading — YOLOv11

### Classes (4):
| ID | Class | Severity Grade |
|---|---|---|
| 0 | corrosion | Grade 1 — Mild |
| 1 | moderate corrosion | Grade 2 — Moderate |
| 2 | rust | Grade 1 — Mild |
| 3 | severe corrosion | Grade 3 — Severe |

## Step 1 — Install Dependencies

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

## Step 2 — Discover Actual Dataset Structure

Run this cell first — it prints **exactly** what Kaggle has mounted, so the path is never wrong.

In [ ]:
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")

print("=" * 60)
print("CONTENTS OF /kaggle/input")
print("=" * 60)

for p in sorted(KAGGLE_INPUT.rglob("*")):
    depth = len(p.relative_to(KAGGLE_INPUT).parts)
    if depth <= 4:   # show up to 4 levels deep
        indent = "  " * (depth - 1)
        tag = "[DIR] " if p.is_dir() else "      "
        print(f"{indent}{tag}{p.name}")

print("=" * 60)
print("\nCopy the correct path above into Step 3 below.")

## Step 3 — Set Paths & Verify Dataset

After running Step 2, set `DATASET_ROOT` to the folder that contains `train/`, `valid/`, `test/`.

In [ ]:
import os, yaml, shutil
from pathlib import Path

# ----------------------------------------------------------
# AUTO-DETECT: walks /kaggle/input and finds the folder
# that contains train/images — no hardcoded slug needed.
# ----------------------------------------------------------
KAGGLE_INPUT = Path("/kaggle/input")
WORK_DIR     = Path("/kaggle/working")

DATASET_ROOT = None
for candidate in KAGGLE_INPUT.rglob("train"):
    if (candidate / "images").is_dir():
        DATASET_ROOT = candidate.parent   # folder containing train/ valid/ test/
        break

if DATASET_ROOT is None:
    raise FileNotFoundError(
        "Could not find a folder containing 'train/images' anywhere under /kaggle/input.\n"
        "Make sure you attached the dataset to this notebook via the 'Add Data' button."
    )

print(f"Auto-detected DATASET_ROOT : {DATASET_ROOT}")
print(f"Working dir                : {WORK_DIR}")
print()

# Count images & labels per split
for split in ['train', 'valid', 'test']:
    img_dir = DATASET_ROOT / split / 'images'
    lbl_dir = DATASET_ROOT / split / 'labels'
    imgs = list(img_dir.glob('*.*')) if img_dir.exists() else []
    lbls = list(lbl_dir.glob('*.txt')) if lbl_dir.exists() else []
    status = "OK" if imgs else "MISSING"
    print(f"  [{status}] {split}: {len(imgs):>5} images,  {len(lbls):>5} labels")

## Step 4 — Create Corrected data.yaml with Absolute Paths

The original `data.yaml` uses relative paths (`../train/images`) which fail on Kaggle's read-only FS.  
We write a corrected version with **absolute paths** into `/kaggle/working/`.

In [ ]:
data_yaml = {
    'train': str(DATASET_ROOT / 'train' / 'images'),
    'val':   str(DATASET_ROOT / 'valid' / 'images'),
    'test':  str(DATASET_ROOT / 'test'  / 'images'),
    'nc': 4,
    'names': ['corrosion', 'moderate corrosion', 'rust', 'severe corrosion']
}

DATA_YAML_PATH = WORK_DIR / 'data.yaml'
with open(DATA_YAML_PATH, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"Written: {DATA_YAML_PATH}")
print()
print(open(DATA_YAML_PATH).read())

## Step 5 — Analyze Class Distribution (Imbalance Check)

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

CLASS_NAMES = ['corrosion', 'moderate corrosion', 'rust', 'severe corrosion']
SEVERITY_MAP = {
    0: ('Grade 1 — Mild',     '#4CAF50'),
    1: ('Grade 2 — Moderate', '#FF9800'),
    2: ('Grade 1 — Mild',     '#8BC34A'),
    3: ('Grade 3 — Severe',   '#F44336')
}

class_counts = Counter()
bbox_sizes   = {i: [] for i in range(4)}

for lbl_file in (DATASET_ROOT / 'train' / 'labels').glob('*.txt'):
    for line in lbl_file.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) >= 5:
            cls = int(parts[0])
            w, h = float(parts[3]), float(parts[4])
            class_counts[cls] += 1
            bbox_sizes[cls].append(w * h)

colors = [SEVERITY_MAP[i][1] for i in range(4)]
labels = [f"{CLASS_NAMES[i]}\n({SEVERITY_MAP[i][0]})" for i in range(4)]
counts = [class_counts[i] for i in range(4)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(labels, counts, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_title('Class Distribution (Training Set)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Annotations')
for i, v in enumerate(counts):
    axes[0].text(i, v + max(counts) * 0.02, str(v), ha='center', fontweight='bold')

box_data = [bbox_sizes[i] for i in range(4)]
bp = axes[1].boxplot(box_data, labels=[CLASS_NAMES[i] for i in range(4)], patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
axes[1].set_title('Bounding Box Area Distribution', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Normalized Area (w x h)')

plt.tight_layout()
plt.savefig(WORK_DIR / 'class_distribution.png', dpi=150)
plt.show()

print("\nClass annotation counts:")
for i in range(4):
    print(f"  {CLASS_NAMES[i]:<22}: {class_counts[i]}")

## Step 6 — Configure Training Hyperparameters

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11m.pt')

TRAIN_CONFIG = dict(
    data          = str(DATA_YAML_PATH),
    epochs        = 100,
    imgsz         = 640,
    batch         = 16,
    patience      = 20,
    optimizer     = 'AdamW',
    lr0           = 0.001,
    lrf           = 0.01,
    weight_decay  = 0.0005,
    warmup_epochs    = 5,
    warmup_momentum  = 0.8,
    warmup_bias_lr   = 0.1,
    hsv_h        = 0.015,
    hsv_s        = 0.7,
    hsv_v        = 0.4,
    degrees      = 15.0,
    translate    = 0.1,
    scale        = 0.5,
    fliplr       = 0.5,
    flipud       = 0.1,
    mosaic       = 1.0,
    mixup        = 0.15,
    copy_paste   = 0.1,
    cos_lr       = True,
    close_mosaic = 15,
    amp          = True,
    project      = str(WORK_DIR / 'runs'),
    name         = 'corrosion_severity',
    exist_ok     = True,
    save         = True,
    save_period  = 25,
    plots        = True,
    verbose      = True,
)

print("Model loaded & config ready")
print(f"  Model   : YOLOv11m ({sum(p.numel() for p in model.model.parameters())/1e6:.1f}M params)")
print(f"  Epochs  : {TRAIN_CONFIG['epochs']}")
print(f"  Img size: {TRAIN_CONFIG['imgsz']}")
print(f"  Batch   : {TRAIN_CONFIG['batch']}")
print(f"  Data    : {TRAIN_CONFIG['data']}")

## Step 7 — Train the Model

In [ ]:
results = model.train(**TRAIN_CONFIG)

## Step 8 — Evaluate on Validation Set

In [ ]:
best_model = YOLO(str(WORK_DIR / 'runs' / 'corrosion_severity' / 'weights' / 'best.pt'))

val_results = best_model.val(data=str(DATA_YAML_PATH), imgsz=640, batch=16, plots=True)

print("\n" + "="*60)
print("VALIDATION RESULTS")
print("="*60)
print(f"  mAP@50     : {val_results.box.map50:.4f}  ({val_results.box.map50*100:.1f}%)")
print(f"  mAP@50-95  : {val_results.box.map:.4f}  ({val_results.box.map*100:.1f}%)")
print(f"  Precision  : {val_results.box.mp:.4f}  ({val_results.box.mp*100:.1f}%)")
print(f"  Recall     : {val_results.box.mr:.4f}  ({val_results.box.mr*100:.1f}%)")
print("="*60)

print("\nPer-Class Performance:")
print(f"{'Class':<22} {'Precision':>10} {'Recall':>10} {'mAP@50':>10} {'mAP@50-95':>10}")
print("-"*65)
for i, name in enumerate(CLASS_NAMES):
    p    = val_results.box.p[i]    if i < len(val_results.box.p)    else 0
    r    = val_results.box.r[i]    if i < len(val_results.box.r)    else 0
    ap50 = val_results.box.ap50[i] if i < len(val_results.box.ap50) else 0
    ap   = val_results.box.ap[i]   if i < len(val_results.box.ap)   else 0
    print(f"  {name:<20} {p:>9.1%} {r:>9.1%} {ap50:>9.1%} {ap:>9.1%}  [{SEVERITY_MAP[i][0]}]")

## Step 9 — Test Set Evaluation

In [ ]:
test_results = best_model.val(
    data  = str(DATA_YAML_PATH),
    split = 'test',
    imgsz = 640,
    batch = 16,
    plots = True
)

print("TEST SET RESULTS")
print(f"  mAP@50    : {test_results.box.map50:.4f}")
print(f"  mAP@50-95 : {test_results.box.map:.4f}")
print(f"  Precision : {test_results.box.mp:.4f}")
print(f"  Recall    : {test_results.box.mr:.4f}")

## Step 10 — Visualize Predictions with Severity Grades

In [ ]:
import cv2, random

SEVERITY_COLORS = {
    0: ((76,  175,  80), 'MILD'),
    1: ((255, 152,   0), 'MODERATE'),
    2: ((139, 195,  74), 'MILD'),
    3: ((244,  67,  54), 'SEVERE')
}

test_image_dir = DATASET_ROOT / 'test' / 'images'
test_images = sorted(test_image_dir.glob('*.*'))
assert len(test_images) > 0, f"No test images found in {test_image_dir}"
sample_imgs = random.sample(test_images, min(8, len(test_images)))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for idx, img_path in enumerate(sample_imgs):
    preds = best_model.predict(str(img_path), imgsz=640, conf=0.25, verbose=False)
    img   = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    for box in preds[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        cls  = int(box.cls[0])
        conf = float(box.conf[0])
        color, severity = SEVERITY_COLORS[cls]
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        label = f"{CLASS_NAMES[cls]} [{severity}] {conf:.0%}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.4, 1)
        cv2.rectangle(img, (x1, y1-th-6), (x1+tw+4, y1), color, -1)
        cv2.putText(img, label, (x1+2, y1-4), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,255,255), 1)
    axes[idx].imshow(img)
    axes[idx].axis('off')

plt.suptitle('Corrosion Detection with Severity Grading', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(WORK_DIR / 'severity_predictions.png', dpi=150)
plt.show()

## Step 11 — Generate Severity Report

In [ ]:
def generate_severity_report(model, image_path):
    preds = model.predict(str(image_path), imgsz=640, conf=0.25, verbose=False)
    if len(preds[0].boxes) == 0:
        return {'overall_grade': 'NONE', 'detections': []}

    grade_scores = {'MILD': 1, 'MODERATE': 2, 'SEVERE': 3}
    detections, max_score, total_area = [], 0, 0
    img_area = 640 * 640

    for box in preds[0].boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        cls      = int(box.cls[0])
        conf     = float(box.conf[0])
        severity = SEVERITY_COLORS[cls][1]
        area     = (x2 - x1) * (y2 - y1)
        total_area += area
        max_score   = max(max_score, grade_scores[severity])
        detections.append({
            'class': CLASS_NAMES[cls], 'severity': severity,
            'confidence': conf, 'area_pct': (area / img_area) * 100
        })

    coverage = (total_area / img_area) * 100
    if   max_score == 3 or coverage > 50: overall = 'CRITICAL'
    elif max_score == 2 or coverage > 25: overall = 'MODERATE'
    else:                                 overall = 'MILD'

    return {
        'overall_grade': overall,
        'num_detections': len(detections),
        'coverage_pct': round(coverage, 1),
        'max_severity': ['MILD', 'MODERATE', 'SEVERE'][max_score - 1],
        'detections': detections
    }

sample = random.choice(test_images)
report = generate_severity_report(best_model, sample)

print("=" * 50)
print(f"SEVERITY REPORT: {Path(sample).name}")
print("=" * 50)
print(f"  Overall Grade   : {report['overall_grade']}")
print(f"  Detections      : {report.get('num_detections', 0)}")
print(f"  Coverage        : {report.get('coverage_pct', 0)}%")
print(f"  Max Severity    : {report.get('max_severity', 'N/A')}")
for i, d in enumerate(report['detections'], 1):
    print(f"    {i}. {d['class']} [{d['severity']}] — {d['confidence']:.0%} conf, {d['area_pct']:.1f}% area")

## Step 12 — Export & Download Best Weights

In [ ]:
best_pt = WORK_DIR / 'runs' / 'corrosion_severity' / 'weights' / 'best.pt'
last_pt = WORK_DIR / 'runs' / 'corrosion_severity' / 'weights' / 'last.pt'

shutil.copy2(best_pt, WORK_DIR / 'best.pt')
shutil.copy2(last_pt, WORK_DIR / 'last.pt')

best_model.export(format='onnx', imgsz=640, simplify=True)

print("Weights saved to /kaggle/working/")
print(f"  best.pt : {best_pt.stat().st_size / 1e6:.1f} MB")
print(f"  last.pt : {last_pt.stat().st_size / 1e6:.1f} MB")
print("\nDownload best.pt from the Output tab on the right.")